In [1]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import joblib

# Load the embedding model
# This model converts sentences into numbers (vectors) that capture meaning
model_embed = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\shrey\OneDrive\Desktop\festiva-planner\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shrey\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [2]:
# Our knowledge base — real event planning knowledge
knowledge_base = [
    # Wedding
    "A typical wedding timeline starts with venue booking 12 months before, catering 8 months before, photography 6 months before, and decoration 2 months before the wedding date.",
    "Wedding budget in India typically splits as: venue 30%, catering 25%, decoration 20%, photography 15%, and miscellaneous 10%.",
    "For a wedding in Bangalore, popular venues include hotels in Indiranagar, Koramangala and Whitefield. Average cost for 200 guests is between 8 to 15 lakhs.",
    "Wedding checklist must include: venue, catering, decoration, photographer, videographer, mehendi artist, music and DJ, bridal wear, and invitation cards.",
    "Outdoor weddings require backup plans for rain, extra lighting arrangements, and generator backup for power.",

    # Corporate
    "A corporate event planning checklist includes: venue selection, audio visual equipment, catering, guest list, agenda planning, and post event feedback.",
    "Corporate event budgets typically allocate 40% for venue, 30% for catering, 15% for AV equipment, and 15% for miscellaneous expenses.",
    "For a successful product launch event, plan 3 months in advance, hire a professional MC, arrange live demonstrations, and prepare press kits.",
    "Team building events work best with outdoor activities, group challenges, and informal networking sessions over meals.",
    "Corporate conferences need reliable WiFi, projectors, microphones, breakout rooms, and catering for tea and lunch breaks.",

    # Birthday
    "A birthday party timeline: send invites 3 weeks before, book venue 1 month before, order cake 1 week before, arrange decorations 1 day before.",
    "Kids birthday party essentials include: return gifts, games and activities, themed decoration, birthday cake, and a designated play area.",
    "Birthday party budget breakdown: venue 35%, catering 30%, decoration 20%, cake 10%, and entertainment 5%.",
    "Surprise birthday parties require a trusted coordinator, a distraction plan, and guests who can keep secrets.",
    "Popular birthday party themes in India include Bollywood, superheroes, princesses, outdoor adventure, and retro themes.",

    # Anniversary
    "Anniversary celebration ideas include: romantic dinner at a rooftop restaurant, destination trip, surprise party with close friends and family.",
    "25th silver anniversary traditionally features silver decorations, a photo slideshow of memories, and renewal of wedding vows.",
    "Anniversary party planning timeline: book venue 2 months before, send invites 1 month before, arrange flowers and cake 1 week before.",
    "For an intimate anniversary dinner, choose a private dining room, arrange flowers and candles, hire a live musician for background music.",
    "Anniversary budget typically splits as: venue and food 50%, decoration 25%, gifts and mementos 15%, photography 10%.",

    # Party
    "House warming party essentials: invite neighbours and close friends, arrange housewarming gifts display, prepare home cooked food or hire a caterer.",
    "New year party planning: book venue 3 months in advance as December dates fill up fast, arrange countdown timer, DJ, and midnight snacks.",
    "Farewell party checklist includes: personalized gifts, memory book with messages from colleagues, favourite food of the person leaving.",
    "Graduation party ideas include outdoor barbecue, photo booth with props, slideshow of college memories, and a customized graduation cake.",
    "Reunion party planning tips: create a WhatsApp group for coordination, collect contributions digitally, arrange a photographer, and prepare a quiz about shared memories.",
]

print(f"Knowledge base created with {len(knowledge_base)} entries!")

Knowledge base created with 25 entries!


In [3]:
# Convert all knowledge base entries into vectors
print("Converting knowledge base to vectors...")
embeddings = model_embed.encode(knowledge_base)

# Store vectors in FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print(f"Total entries indexed: {index.ntotal}")
print(f"Vector dimension: {dimension}")

# Save for later use
faiss.write_index(index, '../models/knowledge_index.faiss')
joblib.dump(knowledge_base, '../models/knowledge_base.pkl')
print("Knowledge base saved!")

Converting knowledge base to vectors...
Total entries indexed: 25
Vector dimension: 384
Knowledge base saved!


In [4]:
def search_knowledge(query, top_k=3):
    # Convert the user's question into a vector
    query_vector = model_embed.encode([query])
    
    # Search FAISS for the most similar entries
    distances, indices = index.search(np.array(query_vector), top_k)
    
    # Return the matching knowledge base entries
    results = []
    for i in indices[0]:
        results.append(knowledge_base[i])
    return results

# Test with real user questions
questions = [
    "What is the best timeline for a wedding?",
    "How should I split my corporate event budget?",
    "What do I need for a kids birthday party?",
]

for q in questions:
    print(f"Question: {q}")
    print("Top answer:")
    print(search_knowledge(q)[0])
    print()

Question: What is the best timeline for a wedding?
Top answer:
A typical wedding timeline starts with venue booking 12 months before, catering 8 months before, photography 6 months before, and decoration 2 months before the wedding date.

Question: How should I split my corporate event budget?
Top answer:
Corporate event budgets typically allocate 40% for venue, 30% for catering, 15% for AV equipment, and 15% for miscellaneous expenses.

Question: What do I need for a kids birthday party?
Top answer:
Kids birthday party essentials include: return gifts, games and activities, themed decoration, birthday cake, and a designated play area.

